In [2]:
!pip install requests python-dotenv --quiet

In [32]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("CUACA_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


In [35]:
alamat_api = "https://api.weatherapi.com/v1/forecast.json"

parameter = {
    "key": API_KEY,
    "q": "Jakarta",
    "days": 3,
    "aqi": "no",
    "alerts": "no"
}

response = requests.get(alamat_api, params=parameter, timeout=20)

print(f"Status Code: {response.status_code}")

hasil = response.json()

if response.status_code != 200:
    print("Pesan error:", hasil)
else:
    print("Lokasi:", hasil["location"]["name"], ",", hasil["location"]["country"])
    print("Jumlah hari forecast:", len(hasil["forecast"]["forecastday"]))


Status Code: 200
Lokasi: Jakarta , Indonesia
Jumlah hari forecast: 3


In [36]:
hari_pertama = hasil["forecast"]["forecastday"][0]
hari_pertama

{'date': '2026-09-23',
 'date_epoch': 1790121600,
 'day': {'maxtemp_c': 33.0,
  'maxtemp_f': 91.4,
  'mintemp_c': 27.2,
  'mintemp_f': 81.0,
  'avgtemp_c': 29.4,
  'avgtemp_f': 84.9,
  'maxwind_mph': 13.0,
  'maxwind_kph': 20.9,
  'totalprecip_mm': 1.34,
  'totalprecip_in': 0.05,
  'totalsnow_cm': 0.0,
  'avgvis_km': 10.0,
  'avgvis_miles': 6.0,
  'avghumidity': 63,
  'daily_will_it_rain': 0,
  'daily_chance_of_rain': 57,
  'daily_will_it_snow': 0,
  'daily_chance_of_snow': 0,
  'condition': {'text': 'Patchy rain nearby',
   'icon': '//cdn.weatherapi.com/weather/64x64/day/176.png',
   'code': 1063},
  'uv': 9.2,
  'avgwetbulb_c': 23.7,
  'avgwetbulb_f': 74.7,
  'maxwetbulb_c': 24.5,
  'maxwetbulb_f': 76.2},
 'astro': {'sunrise': '05:41 AM',
  'sunset': '05:48 PM',
  'moonrise': '03:23 PM',
  'moonset': '03:15 AM',
  'moon_phase': 'Waxing Gibbous',
  'moon_illumination': 95,
  'is_moon_up': 0,
  'is_sun_up': 1},
 'hour': [{'time_epoch': 1790096400,
   'time': '2026-09-23 00:00',
   'tem

In [37]:
jam_pertama = hari_pertama["hour"][0]

print("Waktu       :", jam_pertama["time"])
print("Suhu (°C)   :", jam_pertama["temp_c"])
print("Kondisi     :", jam_pertama["condition"]["text"])
print("Kelembapan  :", jam_pertama["humidity"])
print("Peluang hujan:", jam_pertama["chance_of_rain"])


Waktu       : 2026-09-23 00:00
Suhu (°C)   : 28.2
Kondisi     : Partly Cloudy
Kelembapan  : 66
Peluang hujan: 7


In [38]:
class KlienWeatherAPI:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://api.weatherapi.com/v1"

    def _request(self, endpoint, params):
        params = params.copy()
        params["key"] = self.api_key

        try:
            response = requests.get(
                f"{self.base_url}/{endpoint}",
                params=params,
                timeout=20
            )
            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"Gagal menghubungi WeatherAPI: {e}")
            return None

    def ambil_forecast(self, lokasi, hari=3):
        if not 1 <= hari <= 14:
            raise ValueError("Jumlah hari forecast harus antara 1 dan 14.")

        params = {
            "q": lokasi,
            "days": hari,
            "aqi": "no",
            "alerts": "no"
        }

        return self._request("forecast.json", params)

    def forecast_ke_tabel(self, data):
        if not data:
            return pd.DataFrame()

        baris = []

        nama_lokasi = data["location"]["name"]
        negara = data["location"]["country"]
        latitude = data["location"]["lat"]
        longitude = data["location"]["lon"]

        for hari in data["forecast"]["forecastday"]:
            tanggal = hari["date"]

            for jam in hari["hour"]:
                baris.append({
                    "Lokasi": nama_lokasi,
                    "Negara": negara,
                    "Latitude": latitude,
                    "Longitude": longitude,
                    "Tanggal": tanggal,
                    "Waktu": jam["time"],
                    "Suhu_C": jam["temp_c"],
                    "Terasa_C": jam["feelslike_c"],
                    "Kondisi": jam["condition"]["text"],
                    "Kelembapan_pct": jam["humidity"],
                    "Peluang_Hujan_pct": jam["chance_of_rain"],
                    "Curah_Hujan_mm": jam["precip_mm"],
                    "Kecepatan_Angin_kph": jam["wind_kph"],
                    "Tekanan_mb": jam["pressure_mb"],
                    "Tutupan_Awan_pct": jam["cloud"],
                    "UV": jam["uv"]
                })

        return pd.DataFrame(baris)


In [39]:
klien = KlienWeatherAPI(API_KEY)

daftar_lokasi = [
    "Jakarta",
    "Bandung",
    "Surabaya",
    "Yogyakarta",
    "Denpasar"
]

semua_tabel = []

for lokasi in daftar_lokasi:
    data_forecast = klien.ambil_forecast(lokasi, hari=3)

    if data_forecast:
        tabel = klien.forecast_ke_tabel(data_forecast)
        print(f"{lokasi}: {len(tabel)} baris")
        semua_tabel.append(tabel)

    time.sleep(1)

df_cuaca = pd.concat(semua_tabel, ignore_index=True)

print()
print(f"Total data terkumpul: {len(df_cuaca)} baris")

df_cuaca.head()


Jakarta: 72 baris
Bandung: 72 baris
Surabaya: 72 baris
Yogyakarta: 72 baris
Denpasar: 72 baris

Total data terkumpul: 360 baris


,Lokasi,Negara,Latitude,Longitude,Tanggal,Waktu,Suhu_C,Terasa_C,Kondisi,Kelembapan_pct,Peluang_Hujan_pct,Curah_Hujan_mm,Kecepatan_Angin_kph,Tekanan_mb,Tutupan_Awan_pct,UV
0,Jakarta,Indonesia,-6.2146,106.8451,2026-09-23,2026-09-23 00:00,28.2,30.5,Partly Cloudy,66,7,0.0,6.5,1013.0,35,0.0
1,Jakarta,Indonesia,-6.2146,106.8451,2026-09-23,2026-09-23 01:00,27.9,30.3,Severe smog,68,11,0.0,5.0,1012.0,66,0.0
2,Jakarta,Indonesia,-6.2146,106.8451,2026-09-23,2026-09-23 02:00,27.6,29.8,Severe smog,69,17,0.0,5.4,1012.0,99,0.0
3,Jakarta,Indonesia,-6.2146,106.8451,2026-09-23,2026-09-23 03:00,27.5,29.6,Severe smog,70,16,0.0,5.4,1011.0,94,0.0
4,Jakarta,Indonesia,-6.2146,106.8451,2026-09-23,2026-09-23 04:00,27.3,29.4,Severe smog,71,6,0.0,4.0,1011.0,5,0.0


In [40]:
print("1. Jumlah sel kosong per kolom:")
print(df_cuaca.isnull().sum())
print()

print("2. Jumlah baris kembar berdasarkan Lokasi + Waktu:")
print(df_cuaca.duplicated(subset=["Lokasi", "Waktu"]).sum())
print()

print("3. Tipe data setiap kolom:")
print(df_cuaca.dtypes)


1. Jumlah sel kosong per kolom:
Lokasi                 0
Negara                 0
Latitude               0
Longitude              0
Tanggal                0
Waktu                  0
Suhu_C                 0
Terasa_C               0
Kondisi                0
Kelembapan_pct         0
Peluang_Hujan_pct      0
Curah_Hujan_mm         0
Kecepatan_Angin_kph    0
Tekanan_mb             0
Tutupan_Awan_pct       0
UV                     0
dtype: int64

2. Jumlah baris kembar berdasarkan Lokasi + Waktu:
0

3. Tipe data setiap kolom:
Lokasi                     str
Negara                     str
Latitude               float64
Longitude              float64
Tanggal                    str
Waktu                      str
Suhu_C                 float64
Terasa_C               float64
Kondisi                    str
Kelembapan_pct           int64
Peluang_Hujan_pct        int64
Curah_Hujan_mm         float64
Kecepatan_Angin_kph    float64
Tekanan_mb             float64
Tutupan_Awan_pct         int64
UV      

In [41]:
def bersihkan_kondisi(teks):
    if pd.isna(teks):
        return "Tidak diketahui"
    return teks

df_cuaca["Kondisi"] = df_cuaca["Kondisi"].apply(bersihkan_kondisi)

jumlah_sebelum = len(df_cuaca)

df_cuaca = df_cuaca.dropna(subset=["Lokasi", "Waktu"])

print(f"Baris tanpa Lokasi/Waktu yang dibuang: {jumlah_sebelum - len(df_cuaca)}")

print()
print("Sel kosong setelah penanganan:")
print(df_cuaca.isnull().sum())


Baris tanpa Lokasi/Waktu yang dibuang: 0

Sel kosong setelah penanganan:
Lokasi                 0
Negara                 0
Latitude               0
Longitude              0
Tanggal                0
Waktu                  0
Suhu_C                 0
Terasa_C               0
Kondisi                0
Kelembapan_pct         0
Peluang_Hujan_pct      0
Curah_Hujan_mm         0
Kecepatan_Angin_kph    0
Tekanan_mb             0
Tutupan_Awan_pct       0
UV                     0
dtype: int64


In [42]:
jumlah_sebelum_duplikat = len(df_cuaca)

df_bersih = df_cuaca.drop_duplicates(
    subset=["Lokasi", "Waktu"]
).copy()

print(f"Jumlah baris sebelum : {jumlah_sebelum_duplikat}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {jumlah_sebelum_duplikat - len(df_bersih)}")


Jumlah baris sebelum : 360
Jumlah baris sesudah : 360
Baris kembar dibuang : 0


In [43]:
print("Tipe data sebelum:")
print(df_bersih.dtypes)

df_bersih["Waktu"] = pd.to_datetime(df_bersih["Waktu"], errors="coerce")

kolom_numerik = [
    "Latitude",
    "Longitude",
    "Suhu_C",
    "Terasa_C",
    "Kelembapan_pct",
    "Peluang_Hujan_pct",
    "Curah_Hujan_mm",
    "Kecepatan_Angin_kph",
    "Tekanan_mb",
    "Tutupan_Awan_pct",
    "UV"
]

for kolom in kolom_numerik:
    df_bersih[kolom] = pd.to_numeric(
        df_bersih[kolom],
        errors="coerce"
    )

df_bersih = df_bersih.dropna(subset=["Waktu"]).copy()
df_bersih = df_bersih.sort_values(["Lokasi", "Waktu"]).reset_index(drop=True)

print()
print("Tipe data sesudah:")
print(df_bersih.dtypes)

df_bersih.head()


Tipe data sebelum:
Lokasi                     str
Negara                     str
Latitude               float64
Longitude              float64
Tanggal                    str
Waktu                      str
Suhu_C                 float64
Terasa_C               float64
Kondisi                    str
Kelembapan_pct           int64
Peluang_Hujan_pct        int64
Curah_Hujan_mm         float64
Kecepatan_Angin_kph    float64
Tekanan_mb             float64
Tutupan_Awan_pct         int64
UV                     float64
dtype: object

Tipe data sesudah:
Lokasi                            str
Negara                            str
Latitude                      float64
Longitude                     float64
Tanggal                           str
Waktu                  datetime64[us]
Suhu_C                        float64
Terasa_C                      float64
Kondisi                           str
Kelembapan_pct                  int64
Peluang_Hujan_pct               int64
Curah_Hujan_mm                flo

,Lokasi,Negara,Latitude,Longitude,Tanggal,Waktu,Suhu_C,Terasa_C,Kondisi,Kelembapan_pct,Peluang_Hujan_pct,Curah_Hujan_mm,Kecepatan_Angin_kph,Tekanan_mb,Tutupan_Awan_pct,UV
0,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 00:00:00,19.9,22.1,Overcast,87,28,0.0,2.5,1015.0,100,0.0
1,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 01:00:00,19.6,21.7,Smoky haze,88,27,0.0,2.9,1014.0,96,0.0
2,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 02:00:00,19.4,21.5,Smoky haze,88,28,0.0,2.2,1014.0,98,0.0
3,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 03:00:00,19.2,21.2,Smoky haze,88,18,0.0,2.5,1014.0,52,0.0
4,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 04:00:00,19.1,20.6,Smoky haze,87,13,0.0,4.3,1014.0,9,0.0


In [44]:
# Pemeriksaan terakhir sebelum dataset dianggap selesai

print(f"Jumlah baris           : {len(df_bersih)}")
print(f"Sudah lebih dari 100?  : {len(df_bersih) >= 100}")
print(f"Lokasi masih kosong    : {df_bersih['Lokasi'].isnull().sum()}")
print(f"Waktu masih kosong     : {df_bersih['Waktu'].isnull().sum()}")
print(
    "Duplikat Lokasi+Waktu  :",
    df_bersih.duplicated(subset=["Lokasi", "Waktu"]).sum()
)
print(f"Tipe kolom Waktu       : {df_bersih['Waktu'].dtype}")


Jumlah baris           : 360
Sudah lebih dari 100?  : True
Lokasi masih kosong    : 0
Waktu masih kosong     : 0
Duplikat Lokasi+Waktu  : 0
Tipe kolom Waktu       : datetime64[us]


In [45]:
nama_file = "dataset_prakiraan_cuaca_weatherapi.csv"

df_bersih.to_csv(nama_file, index=False)

print(f"Data berhasil disimpan ke file: {nama_file}")

df_cek = pd.read_csv(nama_file)

print(
    f"File terbaca kembali: "
    f"{len(df_cek)} baris, {len(df_cek.columns)} kolom"
)

df_cek.head()


Data berhasil disimpan ke file: dataset_prakiraan_cuaca_weatherapi.csv
File terbaca kembali: 360 baris, 16 kolom


,Lokasi,Negara,Latitude,Longitude,Tanggal,Waktu,Suhu_C,Terasa_C,Kondisi,Kelembapan_pct,Peluang_Hujan_pct,Curah_Hujan_mm,Kecepatan_Angin_kph,Tekanan_mb,Tutupan_Awan_pct,UV
0,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 00:00:00,19.9,22.1,Overcast,87,28,0.0,2.5,1015.0,100,0.0
1,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 01:00:00,19.6,21.7,Smoky haze,88,27,0.0,2.9,1014.0,96,0.0
2,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 02:00:00,19.4,21.5,Smoky haze,88,28,0.0,2.2,1014.0,98,0.0
3,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 03:00:00,19.2,21.2,Smoky haze,88,18,0.0,2.5,1014.0,52,0.0
4,Bandung,Indonesia,-6.9039,107.6186,2026-09-23,2026-09-23 04:00:00,19.1,20.6,Smoky haze,87,13,0.0,4.3,1014.0,9,0.0


In [46]:
print("=" * 60)
print("ANGKA UNTUK SLIDE MINI PROJECT")
print("=" * 60)

print("Sumber data        : WeatherAPI.com")
print("Endpoint           : /forecast.json")
print("Jenis data         : Prakiraan cuaca per jam")
print(f"Lokasi yang dipakai: {', '.join(daftar_lokasi)}")
print()

print(f"Baris sebelum cleaning : {len(df_cuaca)}")
print(f"Baris dataset akhir    : {len(df_bersih)}")
print()

print("Class yang dibuat:")
print("  1. KlienWeatherAPI")
print("     - _request()")
print("     - ambil_forecast()")
print("     - forecast_ke_tabel()")
print()

print("Function cleaning:")
print("  1. bersihkan_kondisi()")
print()

print("Temuan cleaning:")
print(
    "  Baris kembar dibuang :",
    df_cuaca.duplicated(subset=["Lokasi", "Waktu"]).sum()
)
print("  Kondisi kosong diberi penanda: ya")
print()

print("Kolom dataset:")
print(", ".join(df_bersih.columns))

print("=" * 60)


ANGKA UNTUK SLIDE MINI PROJECT
Sumber data        : WeatherAPI.com
Endpoint           : /forecast.json
Jenis data         : Prakiraan cuaca per jam
Lokasi yang dipakai: Jakarta, Bandung, Surabaya, Yogyakarta, Denpasar

Baris sebelum cleaning : 360
Baris dataset akhir    : 360

Class yang dibuat:
  1. KlienWeatherAPI
     - _request()
     - ambil_forecast()
     - forecast_ke_tabel()

Function cleaning:
  1. bersihkan_kondisi()

Temuan cleaning:
  Baris kembar dibuang : 0
  Kondisi kosong diberi penanda: ya

Kolom dataset:
Lokasi, Negara, Latitude, Longitude, Tanggal, Waktu, Suhu_C, Terasa_C, Kondisi, Kelembapan_pct, Peluang_Hujan_pct, Curah_Hujan_mm, Kecepatan_Angin_kph, Tekanan_mb, Tutupan_Awan_pct, UV
